# GAN Defect Augmentation - Phase 1: Setup

This notebook guides you through the complete setup process for the GAN defect augmentation project.

## Step 1: Environment Setup

Create and activate the conda environment:

```bash
conda env create -f environment.yml
conda activate gan-defect-augmentation
```

## Step 2: Verify Installation

In [ ]:
import torch
import torchvision
import albumentations
import wandb
import timm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"Albumentations version: {albumentations.__version__}")
print(f"Timm version: {timm.__version__}")

## Step 3: Download MVTec AD Dataset

Run the download script from terminal:

```bash
python download_mvtec.py
```

This will:
- Download MVTec AD dataset (~5.3 GB)
- Extract to `./data/raw/mvtec/`
- Create train/val/test splits (80/10/10)
- Verify dataset structure

## Step 4: Test Data Loading

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data.mvtec_dataset import MVTecDataLoader
from src.utils.config import load_config
import matplotlib.pyplot as plt
import numpy as np

# Load config
config = load_config('../config.yaml')

# Create data loader for bottle category
data_loader = MVTecDataLoader(
    root_dir=config.data.raw_dir,
    category='bottle',
    batch_size=4,
    num_workers=0,
    image_size=config.data.image_size,
)

train_loader, val_loader, test_loader = data_loader.get_loaders()

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## Step 5: Visualize Sample Data

In [ ]:
# Get a batch
batch = next(iter(train_loader))

print(f"Image shape: {batch['image'].shape}")
print(f"Mask shape: {batch['mask'].shape}")
print(f"Label shape: {batch['label'].shape}")
print(f"Labels: {batch['label']}")

# Denormalize for visualization
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(4, 3, figsize=(12, 12))

for i in range(4):
    # Image
    img = batch['image'][i].permute(1, 2, 0).numpy()
    img = (img * std + mean).clip(0, 1)
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"Image {i}")
    axes[i, 0].axis('off')
    
    # Mask
    mask = batch['mask'][i].numpy()
    axes[i, 1].imshow(mask, cmap='gray')
    axes[i, 1].set_title(f"Mask {i}")
    axes[i, 1].axis('off')
    
    # Label
    label = batch['label'][i].item()
    axes[i, 2].text(0.5, 0.5, f"Label: {label}\n(0=normal, 1=defect)", 
                    ha='center', va='center', fontsize=12)
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('../outputs/sample_data.png', dpi=150, bbox_inches='tight')
plt.show()

print("Sample visualization saved to outputs/sample_data.png")

## Step 6: Dataset Statistics

In [ ]:
import pandas as pd

# Collect statistics for all categories
stats = []

for category in config.data.categories:
    try:
        loader = MVTecDataLoader(
            root_dir=config.data.raw_dir,
            category=category,
            batch_size=1,
            num_workers=0,
        )
        train_loader, val_loader, test_loader = loader.get_loaders()
        
        stats.append({
            'Category': category,
            'Train': len(train_loader),
            'Val': len(val_loader),
            'Test': len(test_loader),
            'Total': len(train_loader) + len(val_loader) + len(test_loader),
        })
    except Exception as e:
        print(f"Error loading {category}: {e}")

df_stats = pd.DataFrame(stats)
print(df_stats)
print(f"\nTotal images: {df_stats['Total'].sum()}")

## Next Steps

Phase 1 setup is complete! You now have:

✅ Conda environment with all dependencies
✅ MVTec AD dataset downloaded and organized
✅ Data loading pipeline tested
✅ Sample data visualization

**Next: Phase 2 - GAN Architecture & Training**

Run the GAN training script:
```bash
python src/train_gan.py --config config.yaml
```